# Software specs, RTL facts and CCA

A software spec says what a computation means. RTL facts describe what a selected
hardware source contains. This notebook loads a small spec, inspects a bit slice
in HW-dialect text, and shows what happens when the required facts are missing.

All inputs here are authored teaching examples. We run Merlin's real readers,
but we do not elaborate a target, execute instructions or measure performance.
See [the guide](../../docs/guides/02-targets.md) for the real extraction commands.
Use the shared [tutorial launcher](README.md) to open this notebook.

**Step 1 of 5: Contracts and facts** | [All tutorials](README.md) | [Next: Test generation](02_test_generation.ipynb)


In [ ]:
import json
from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory

from merlin.target.spec import load_spec
from merlin.development.targetgen.software_spec import (
    admit_operation, capability_contract, numerical_datapath,
)
from merlin.development.targetgen.rtl.facts import (
    FactsEmpty, facts_body, load_facts, unknown_reasons, validate_facts,
)
from merlin.development.targetgen.rtl.hw_graph import parse_generic_hw
from merlin.development.targetgen.rtl.hw_observations import input_observations
from merlin.analysis.kernels.cca import CCA
from merlin.analysis.kernels.cca_compare import compare_evidence

## 1. Load a software spec

This draft describes signed integer contraction inputs and an i32 accumulator.
The `operations` entry restricts the supported input type, rank and layout.
The loader keeps the source identity separate from its normalized consumer view.

In [ ]:
software_yaml = """
schema: merlin.software_spec.v1
target: tutorial-array
status: unreviewed
numerical_semantics:
  model: {engine: integer_reference}
  operand_dtype: int8
  accumulator_dtype: i32
  readout_dtype: i32
  subnormal_operand_flush: false
operations:
  contraction:
    families: [contraction]
    placement: accelerator
    operand_dtypes: [int8]
    accumulator_dtype: i32
    ranks: [2]
    layouts: [row_major_contiguous]
"""
workspace = TemporaryDirectory(prefix="merlin-contracts-tutorial-")
spec_path = Path(workspace.name) / "software-spec.yaml"
spec_path.write_text(software_yaml)
expansion = load_spec(spec_path, target="tutorial-array")
spec = expansion.software_spec()
record = expansion.record()
pprint({key: record[key] for key in (
    "source_schema", "target", "status", "projection", "qualification", "source_sha256",
)})
print("Declared numerical types:", numerical_datapath(spec)["operand_dtype"],
      "->", numerical_datapath(spec)["accum_dtype"])

In [ ]:
def screen(operand_dtype="int8", rank=2):
    signature = {
        "family": "contraction", "operand_dtype": operand_dtype,
        "accum_dtype": "i32", "rank": rank, "layout": "row_major_contiguous",
    }
    return admit_operation(spec, "matmul", signature, "accelerator")

pprint(screen())
pprint(screen(operand_dtype="f32"))

A matching request stays `unknown` because this draft is unreviewed. A conflicting
type is `unsupported`. This is software admission; hardware and implemented lowering
have their own checks.

The backend contract is selected separately. The function below checks that it
names the same target; it does not fill in an ISA or memory map from the spec.

In [ ]:
pprint(capability_contract(spec, base_contract={"name": "tutorial-array"}))
try:
    capability_contract(spec, base_contract={"name": "another-target"})
except ValueError as error:
    print(error)

## 2. Observe a small HW graph

The input is an 8-bit wire. The graph takes four bits and compares them with 3.
`parse_generic_hw` reads the generic MLIR form; `input_observations` follows the
slice back to the input. It reports the bit range and equality constant.

An 8-bit wire could carry many encodings. This local observation does not tell us
whether it is signed int8, a floating format, an opcode or tensor data.

In [ ]:
def hardware_text(bits=8, low=0):
    return f"""builtin.module {{
  "hw.module"() ({{
  ^bb0(%data: i{bits}):
    %part = "comb.extract"(%data) {{lowBit = {low} : i32}} : (i{bits}) -> i4
    %expected = "hw.constant"() {{value = 3 : i4}} : () -> i4
    %equal = "comb.icmp"(%part, %expected) {{predicate = 0 : i64}} : (i4, i4) -> i1
    "hw.output"(%equal) : (i1) -> ()
  }}) {{sym_name = "Tutorial", module_type = !hw.modty<input data: i{bits}, output result: i1>}} : () -> ()
}}"""

pprint(input_observations(parse_generic_hw(hardware_text())))

## 3. Keep unavailable facts visible

A complete target extraction also needs source/configuration identities and the
observations required by its consumers. To show the missing-input case, load an
explicitly authored facts packet with an empty body and a recorded reason.
`regenerate=False` keeps this read from starting RTL extraction.

In [ ]:
facts_path = Path(workspace.name) / "unknown-facts.json"
facts_path.write_text(json.dumps({
    "schema_version": "2.0",
    "family": "circt_static",
    "facts": {},
    "unknown": {"target_rtl": "No target RTL/elaboration was supplied in this tutorial."},
    "provenance": {"kind": "authored tutorial packet", "qualification": "not_established"},
}))
selected = load_facts("tutorial-array", explicit=facts_path, regenerate=False)
print("Schema problems:", validate_facts(selected, target="tutorial-array"))
print("Recorded unknowns:", unknown_reasons(selected))
try:
    facts_body(selected, "tutorial-array", needs="a real scheduling resource description")
except FactsEmpty as error:
    print(type(error).__name__ + ":", str(error).split(" Artifact:", 1)[0])

The packet has a valid shape, yet its empty body cannot supply scheduling resources.
Validation of the file format and availability of hardware facts answer different
questions.

## 4. See why CCA is another step

CCA describes a selected program, instruction stream or schedule. A declaration
that expects i32 accumulation cannot fill an *observed* CCA accumulator field.
The role-based lifter needs decoded commands and their endpoint; the vector lifter
needs a decoded vector stream. Here we have neither.

In [ ]:
unobserved = CCA(op="matmul", backend=[spec["target"]])
print("Declared accumulator:", numerical_datapath(spec)["accum_dtype"])
print("Observed CCA accumulator:", unobserved.compute.accumulator_dtype)
comparison = compare_evidence(unobserved, unobserved)
print("Comparison:", comparison.status, "with", comparison.denominator, "comparable axes")

## 5. Change the request and the bit slice

Try rank 3 or f32 to change software admission. Then change the wire width or
slice offset. Notice that changing the HW text does not silently rewrite the
software spec. A real integration needs a checked correspondence between them.

In [ ]:
def inspect_contracts(operand_dtype="int8", rank=2, bits=8, low=0):
    from html import escape
    from IPython.display import HTML, display

    numeric = numerical_datapath(spec)
    admission = screen(operand_dtype, rank)
    observed = input_observations(parse_generic_hw(hardware_text(bits, low)))
    module = observed["modules"][0]
    wire = module["inputs"][0]
    part = module["observations"][0]
    left = [
        ("Spec status", spec["status"]),
        ("Operand type", numeric["operand_dtype"]),
        ("Accumulator type", numeric["accum_dtype"]),
        ("Request", operand_dtype + ", rank " + str(rank)),
        ("Software admission", admission["status"]),
    ]
    right = [
        ("Input", wire["name"]),
        ("Wire width", wire["width"]),
        ("Slice offset", part["offset"]),
        ("Slice width", part["width"]),
        ("Equality constants", part["equality_constants"]),
        ("Complete ISA", observed["complete_isa"]),
    ]

    def table(title, rows):
        return "<section><h4>" + escape(title) + "</h4><table>" + "".join(
            "<tr><th>" + escape(label) + "</th><td>" + escape(str(value)) + "</td></tr>"
            for label, value in rows
        ) + "</table></section>"

    bit_cells = "".join(
        "<span style='background:" + ("#b9def7" if part["offset"] <= bit < part["offset"] + part["width"] else "#eee")
        + ";color:#17212b;display:inline-block;min-width:25px;padding:5px;margin:1px;text-align:center'>"
        + str(bit) + "</span>" for bit in reversed(range(wire["width"]))
    )
    display(HTML(
        "<style>.contracts-view{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:18px}"
        ".contracts-view th,.contracts-view td{padding:6px;text-align:left;border-bottom:1px solid #aaa}</style>"
        "<div class='contracts-view'>" + table("Software declaration and request", left)
        + table("Parsed local HW observation", right) + "</div>"
        "<p><b>Input bits</b> (high to low; blue is the observed slice):</p>" + bit_cells
    ))

inspect_contracts()

In [ ]:
try:
    import ipywidgets as widgets
except ImportError:
    print("Call inspect_contracts(operand_dtype='f32', bits=16, low=4) to explore without widgets.")
else:
    from IPython.display import display

    controls = widgets.interactive(
        inspect_contracts,
        operand_dtype=widgets.Dropdown(options=["int8", "f32"], description="Request type"),
        rank=widgets.Dropdown(options=[2, 3], description="Rank"),
        bits=widgets.Dropdown(options=[8, 16], description="Wire bits"),
        low=widgets.IntSlider(min=0, max=4, value=0, description="Slice offset"),
    )
    display(controls)

## Continue with a real target

Open [the target-contract guide](../../docs/guides/02-targets.md) for the
existing Gemmini/Atlas inputs, extraction commands and source-selection checks.
It follows the selected facts into capability resolution, shared scheduling
resources and the decoder inputs used by CCA.

For executable pass examples, use the [tutorial catalog](README.md). For CCA
comparison and proposed compiler edits, open [cca.ipynb](05_cca.ipynb).

**Step 1 of 5: Contracts and facts** | [All tutorials](README.md) | [Next: Test generation](02_test_generation.ipynb)
